# 07 — ETL FINAL v2: sin `place_country:US`, con filtrado geográfico en pandas

## Por qué este notebook existe

La primera tirada definitiva (notebook 06) usaba `place_country:US` en la query, que requiere geolocalización confirmada por X. Como <1% de tweets tiene esa metadata, solo se obtuvieron 327 brutos de las hasta 20.000 esperadas.

**Cambio principal:**
- Las queries ya **NO** incluyen `place_country:US`.
- El filtrado geográfico se hace **después** en pandas, examinando `user_location` y otras señales (similar al filtro `looks_us()` que ya funcionaba).
- Resultado esperado: 10-50× más volumen.

## Pipeline de este notebook

1. Lanza las 5 queries amplias (sin `place_country:US`).
2. Guarda el corpus crudo de esta tirada en `scam_us_run2_raw.csv`.
3. **Une con el corpus de la primera tirada** (`scam_us_FINAL_raw.csv`) y deduplica.
4. Aplica todos los filtros estructurales (v4) sobre el corpus unido.
5. Guarda corpus consolidado y limpio.

## Coste estimado

- **Hasta 40 llamadas API** (5 queries × 8 páginas).
- Tiempo: 8-15 minutos.


In [1]:
# Imports y carga del token
import os
import time
import requests
import pandas as pd
import re
from dotenv import load_dotenv

pd.set_option('display.max_colwidth', None)

load_dotenv()
TOKEN = os.getenv("X_BEARER_TOKEN", "")
if not TOKEN:
    raise ValueError("Falta X_BEARER_TOKEN en .env")

BASE_URL = "https://api.x.com/2/tweets/search/all"


In [2]:
def search_all_posts_to_df(
    bearer_token, query, max_pages=8, max_results=500,
    sleep_seconds=1.1, start_time=None, end_time=None, label=""
):
    headers = {"Authorization": f"Bearer {bearer_token}"}
    params = {
        "query": query,
        "max_results": max_results,
        "tweet.fields": "id,text,created_at,author_id,lang,geo,public_metrics,entities",
        "expansions": "author_id,geo.place_id",
        "user.fields": "id,name,username,created_at,location,verified,public_metrics",
        "place.fields": "id,full_name,country,country_code,place_type,geo",
        "sort_order": "recency",
    }
    if start_time: params["start_time"] = start_time
    if end_time: params["end_time"] = end_time

    all_rows = []
    next_token = None

    for page in range(max_pages):
        if next_token:
            params["next_token"] = next_token
        else:
            params.pop("next_token", None)

        retries = 0
        while True:
            r = requests.get(BASE_URL, headers=headers, params=params, timeout=30)
            if r.status_code == 200:
                break
            elif r.status_code == 429:
                wait = min(60 * (2 ** retries), 900)
                print(f"  Rate limit. Esperando {wait}s...")
                time.sleep(wait)
                retries += 1
                if retries > 5: raise RuntimeError("Demasiados reintentos")
            elif r.status_code == 400:
                raise RuntimeError(f"400 Bad Request. Query len={len(query)}. {r.text}")
            elif r.status_code == 403:
                raise RuntimeError(f"403 Forbidden. {r.text}")
            else:
                raise RuntimeError(f"X API error {r.status_code}: {r.text}")

        payload = r.json()
        data = payload.get("data", []) or []
        includes = payload.get("includes", {}) or {}
        meta = payload.get("meta", {}) or {}
        users_by_id = {u["id"]: u for u in includes.get("users", []) or []}
        places_by_id = {p["id"]: p for p in includes.get("places", []) or []}

        for t in data:
            author = users_by_id.get(t.get("author_id"), {})
            place_id = (t.get("geo") or {}).get("place_id")
            place = places_by_id.get(place_id, {}) if place_id else {}
            entities = t.get("entities") or {}
            metrics = t.get("public_metrics") or {}
            am = author.get("public_metrics") or {}

            all_rows.append({
                "tweet_id": t.get("id"),
                "created_at": t.get("created_at"),
                "text": t.get("text"),
                "lang": t.get("lang"),
                "author_id": t.get("author_id"),
                "username": author.get("username"),
                "name": author.get("name"),
                "user_location": author.get("location"),
                "user_verified": author.get("verified"),
                "user_followers": am.get("followers_count"),
                "user_tweet_count": am.get("tweet_count"),
                "place_id": place_id,
                "place_full_name": place.get("full_name"),
                "place_country": place.get("country"),
                "place_country_code": place.get("country_code"),
                "place_type": place.get("place_type"),
                "retweet_count": metrics.get("retweet_count"),
                "reply_count": metrics.get("reply_count"),
                "like_count": metrics.get("like_count"),
                "quote_count": metrics.get("quote_count"),
                "n_hashtags": len(entities.get("hashtags", []) or []),
                "n_mentions": len(entities.get("mentions", []) or []),
                "n_urls": len(entities.get("urls", []) or []),
                "query_label": label,
            })

        print(f"  [{label} | p{page+1}/{max_pages}] {len(all_rows)} acumulados")
        next_token = meta.get("next_token")
        if not next_token:
            print(f"  [{label}] sin más páginas")
            break
        time.sleep(sleep_seconds)

    return pd.DataFrame(all_rows)


## Queries SIN `place_country:US`

El filtro geográfico se hará después en pandas usando `user_location`.


In [3]:
EX_MIN = "-trump -biden -election -voter -ballot -nba -nfl -kpop"
# OJO: SIN place_country:US — esa es la diferencia clave
GEO_OPS = "lang:en -is:retweet -is:reply -is:nullcast"

def build(positive):
    q = f"({positive}) {EX_MIN} {GEO_OPS}"
    if len(q) > 512:
        raise ValueError(f"Query excede 512 chars: {len(q)}")
    return q

QUERIES = {
    "phishing": build(
        '"phishing email" OR "phishing text" OR "scam text" '
        'OR "scam email" OR "smishing" OR "got a text from" OR "fake text"'
    ),
    "payment_apps": build(
        '"Zelle scam" OR "Venmo scam" OR "Cash App scam" '
        'OR "PayPal scam" OR "Apple Pay scam"'
    ),
    "crypto": build(
        '"rug pull" OR "pig butchering" OR "crypto scam" '
        'OR "bitcoin scam" OR "investment scam" OR "ponzi scheme"'
    ),
    "romance_monetary": build(
        '("romance scam" OR "dating scam") '
        '(money OR sent OR wire OR transferred)'
    ),
    "impersonation": build(
        '"IRS scam" OR "USPS scam" OR "FedEx scam" OR "toll scam" '
        'OR "fake delivery" OR "fake invoice" '
        'OR "tech support scam" OR "job scam"'
    ),
}

print("=== QUERIES SIN place_country:US ===\n")
for label, q in QUERIES.items():
    print(f"{label:<20} {len(q):>4} chars")
    print(f"  {q}\n")


=== QUERIES SIN place_country:US ===

phishing              216 chars
  ("phishing email" OR "phishing text" OR "scam text" OR "scam email" OR "smishing" OR "got a text from" OR "fake text") -trump -biden -election -voter -ballot -nba -nfl -kpop lang:en -is:retweet -is:reply -is:nullcast

payment_apps          184 chars
  ("Zelle scam" OR "Venmo scam" OR "Cash App scam" OR "PayPal scam" OR "Apple Pay scam") -trump -biden -election -voter -ballot -nba -nfl -kpop lang:en -is:retweet -is:reply -is:nullcast

crypto                204 chars
  ("rug pull" OR "pig butchering" OR "crypto scam" OR "bitcoin scam" OR "investment scam" OR "ponzi scheme") -trump -biden -election -voter -ballot -nba -nfl -kpop lang:en -is:retweet -is:reply -is:nullcast

romance_monetary      172 chars
  (("romance scam" OR "dating scam") (money OR sent OR wire OR transferred)) -trump -biden -election -voter -ballot -nba -nfl -kpop lang:en -is:retweet -is:reply -is:nullcast

impersonation         230 chars
  ("IRS sc

## Confirmación de la tirada

⚠️ Escribe **LANZAR** (todo en mayúsculas) en el cuadro de texto que aparecerá arriba del notebook.


In [4]:
PAGES_PER_QUERY = 8
START = "2025-01-01T00:00:00Z"
END   = "2025-12-31T23:59:59Z"

est_calls = len(QUERIES) * PAGES_PER_QUERY
est_max_tweets = est_calls * 500
print("=" * 60)
print("   SEGUNDA TIRADA DEFINITIVA — SIN place_country:US")
print("=" * 60)
print(f"  Páginas por query:        {PAGES_PER_QUERY}")
print(f"  Categorías:               {len(QUERIES)}")
print(f"  Llamadas API:             hasta {est_calls}")
print(f"  Tweets brutos máx:        {est_max_tweets}")
print(f"  Ventana temporal:         {START} → {END}")
print(f"  Tiempo estimado:          ~8-15 minutos")
print("=" * 60)
print()
print("⚠️ Filtrado geográfico se hará DESPUÉS en pandas (user_location).")
print()
confirmation = input("Escribe LANZAR para ejecutar, cualquier otra cosa para abortar: ")
if confirmation.strip().upper() != "LANZAR":
    raise RuntimeError("Tirada abortada por el usuario.")
print("\n✓ Confirmación recibida. Iniciando...\n")


   SEGUNDA TIRADA DEFINITIVA — SIN place_country:US
  Páginas por query:        8
  Categorías:               5
  Llamadas API:             hasta 40
  Tweets brutos máx:        20000
  Ventana temporal:         2025-01-01T00:00:00Z → 2025-12-31T23:59:59Z
  Tiempo estimado:          ~8-15 minutos

⚠️ Filtrado geográfico se hará DESPUÉS en pandas (user_location).


✓ Confirmación recibida. Iniciando...



## Ejecución

In [5]:
dfs = []
for label, query in QUERIES.items():
    print(f"\n>>> {label}")
    df_part = search_all_posts_to_df(
        bearer_token=TOKEN, query=query,
        max_pages=PAGES_PER_QUERY, max_results=500,
        sleep_seconds=1.1, start_time=START, end_time=END,
        label=label,
    )
    print(f">>> {label}: {len(df_part)}")
    dfs.append(df_part)

df_run2 = pd.concat(dfs, ignore_index=True)
print(f"\n=== TOTAL BRUTO RUN 2: {len(df_run2)} ===")
print("\nPor categoría:")
print(df_run2["query_label"].value_counts())



>>> phishing
  [phishing | p1/8] 472 acumulados
  [phishing | p2/8] 933 acumulados
  [phishing | p3/8] 1411 acumulados
  [phishing | p4/8] 1882 acumulados
  [phishing | p5/8] 2357 acumulados
  [phishing | p6/8] 2818 acumulados
  [phishing | p7/8] 3282 acumulados
  [phishing | p8/8] 3757 acumulados
>>> phishing: 3757

>>> payment_apps
  [payment_apps | p1/8] 415 acumulados
  [payment_apps | p2/8] 794 acumulados
  [payment_apps] sin más páginas
>>> payment_apps: 794

>>> crypto
  [crypto | p1/8] 472 acumulados
  [crypto | p2/8] 935 acumulados
  [crypto | p3/8] 1398 acumulados
  [crypto | p4/8] 1867 acumulados
  [crypto | p5/8] 2353 acumulados
  [crypto | p6/8] 2827 acumulados
  [crypto | p7/8] 3308 acumulados
  [crypto | p8/8] 3796 acumulados
>>> crypto: 3796

>>> romance_monetary
  [romance_monetary | p1/8] 317 acumulados
  [romance_monetary | p2/8] 491 acumulados
  [romance_monetary | p3/8] 671 acumulados
  [romance_monetary | p4/8] 778 acumulados
  [romance_monetary] sin más páginas


## Guardado inmediato del bruto de la segunda tirada

⚠️ Guardar antes de unir. Si algo va mal con la unión, podemos rehacerla sin tocar la API.


In [6]:
os.makedirs("../data/raw", exist_ok=True)
df_run2.to_csv("../data/raw/scam_us_run2_raw.csv", index=False, encoding="utf-8")
print(f"✓ Guardado: ../data/raw/scam_us_run2_raw.csv ({len(df_run2)} filas)")
print()
print("🚨 HAZ COPIA DE SEGURIDAD EN DRIVE/DROPBOX AHORA mismo.")


✓ Guardado: ../data/raw/scam_us_run2_raw.csv (12419 filas)

🚨 HAZ COPIA DE SEGURIDAD EN DRIVE/DROPBOX AHORA mismo.


## Unión con el corpus de la primera tirada

In [7]:
# Cargar la primera tirada
try:
    df_run1 = pd.read_csv("../data/raw/scam_us_FINAL_raw.csv")
    print(f"Run 1 (con place_country:US): {len(df_run1)} tweets")
except FileNotFoundError:
    print("⚠️  No se encontró scam_us_FINAL_raw.csv — se usará solo la run 2.")
    df_run1 = pd.DataFrame()

print(f"Run 2 (sin place_country:US): {len(df_run2)} tweets")

# Concatenar
df_all = pd.concat([df_run1, df_run2], ignore_index=True)
print(f"\nConcatenado bruto: {len(df_all)}")

# Deduplicar conservando todas las query_labels de cada tweet
df_all["query_labels"] = df_all.groupby("tweet_id")["query_label"].transform(
    lambda s: ",".join(sorted(set(s)))
)
df_dedup = df_all.drop_duplicates(subset="tweet_id", keep="first").reset_index(drop=True)
df_dedup = df_dedup.drop(columns=["query_label"])

print(f"Únicos tras deduplicar entre ambas tiradas: {len(df_dedup)}")
print(f"Multi-categoría: {(df_dedup['query_labels'].str.contains(',')).sum()}")

df_dedup.to_csv("../data/raw/scam_us_CONSOLIDATED_dedup.csv", index=False, encoding="utf-8")
print(f"\n✓ Guardado: ../data/raw/scam_us_CONSOLIDATED_dedup.csv")


Run 1 (con place_country:US): 327 tweets
Run 2 (sin place_country:US): 12419 tweets

Concatenado bruto: 12746
Únicos tras deduplicar entre ambas tiradas: 12726
Multi-categoría: 20

✓ Guardado: ../data/raw/scam_us_CONSOLIDATED_dedup.csv


## Filtros estructurales (los del v4)

Aplica los mismos filtros, pero ahora sobre el corpus unificado.


In [8]:
STRONG_FRAUD_TERMS = re.compile(
    r"\b(scam|scammed|scammer|scammers|fraud|fraudster|defrauded|"
    r"phishing|smishing|vishing|ponzi|rug\s*pull|pig\s*butchering|"
    r"identity\s*theft|wire\s*fraud|impersonat|spoof|"
    r"fake\s*(?:text|email|delivery|invoice|website|caller))\b",
    re.IGNORECASE,
)
MONEY_TERMS = re.compile(
    r"(\b(money|cash|dollar|dollars|usd|paid|payment|sent|wired|wire|"
    r"transfer|refund|bank|account|credit\s*card|debit|wallet|invoice|"
    r"charged|stolen|lost|victim|defrauded|deposit|withdraw|"
    r"check|venmo|zelle|paypal|crypto|bitcoin|coinbase|gift\s*card)\b|"
    r"\$\s*\d+|\d+\s*(k|dollars|usd))",
    re.IGNORECASE,
)
RECOVERY_SCAM_PATTERNS = re.compile(
    r"\b(DM\s+(now|me|us|asap)|we\s+are\s+tracing|get\s+your\s+money\s+back|"
    r"recover(y|\s+expert|\s+specialist)|funds\s+recovery|crypto\s+recovery|"
    r"contact\s+us\s+(?:now|asap|today)|reach\s+out\s+to\s+(?:us|me)|"
    r"100%\s+(?:guaranteed|recovery|success))\b",
    re.IGNORECASE,
)
BRAND_LIST_PATTERN = re.compile(
    r"(nft\s+scam|bitcoin\s+scam|forex|cfd|binary\s+option|"
    r"wallet\s+drained|bored\s*apes?|ethereum\s+scam)",
    re.IGNORECASE,
)
def is_recovery_scammer(text):
    if not isinstance(text, str): return False
    if RECOVERY_SCAM_PATTERNS.search(text): return True
    has_contact = bool(re.search(r"\b(DM|dm)\b|contact|reach\s+out", text, re.IGNORECASE))
    return has_contact and len(BRAND_LIST_PATTERN.findall(text)) >= 2

PROMO_BLOG_PATTERN = re.compile(
    r"\b(our\s+(?:recent\s+)?(?:blog|article|post)|breaks?\s+down|"
    r"read\s+(?:more|the\s+full)|link\s+in\s+bio|"
    r"check\s+(?:out|it)\s+(?:our|this)|"
    r"new\s+(?:blog|article|episode|video))\b",
    re.IGNORECASE,
)
NEWS_HEADLINE_PATTERN = re.compile(
    r"^\W*(leaders?|chief|ceo|president|founder|judge|jury|court|"
    r"police|sentence[d]?|charged|arrested|indicted|"
    r"\d+\s+(?:years?|months?)\s+in\s+prison)\b",
    re.IGNORECASE,
)
def looks_like_news_or_promo(text, n_urls=0):
    if not isinstance(text, str): return False
    if PROMO_BLOG_PATTERN.search(text): return True
    if n_urls > 0 and NEWS_HEADLINE_PATTERN.search(text): return True
    return False

INSTITUTIONAL_TARGETS = [
    "social security", "stock market", "housing market", "real estate market",
    r"\bai\b", "artificial intelligence", "crypto industry",
    "the tax system", "tax system", "taxes", "capitalism", "the government",
    "modern slavery", "slave engine", "marriage", "religion", "the church",
    "college", "the system", "the economy", "the federal reserve",
    "fiat currency", "social welfare", "welfare system",
    "healthcare system", "education system", "school system",
    "pension", "401k", "wall street", "usaid", "dnc",
]
ACCUSATION_WORDS = ["ponzi", "scheme", "scam", "fraud"]
WINDOW = r"(?:\W+\w+){0,6}\W+"
INSTITUTIONAL_METAPHOR_RE = re.compile(
    "|".join(
        f"({inst}{WINDOW}(?:{'|'.join(ACCUSATION_WORDS)}))"
        f"|((?:{'|'.join(ACCUSATION_WORDS)}){WINDOW}{inst})"
        for inst in INSTITUTIONAL_TARGETS
    ),
    re.IGNORECASE,
)
def is_institutional_metaphor(text):
    if not isinstance(text, str): return False
    return bool(INSTITUTIONAL_METAPHOR_RE.search(text))

METAPHOR_PATTERNS = re.compile(
    r"\b(life is a scam|love is a scam|system is a scam|"
    r"capitalism is a scam|college is a scam|dating is a scam|"
    r"my life is a scam|everything is a scam)\b",
    re.IGNORECASE,
)
# Política suave REFORZADA con nombres propios que se nos colaron en run 1
SOFT_POLITICS_PATTERNS = re.compile(
    r"\b(harris|desantis|newsom|kamala|obama|congress|senate|senator|"
    r"republican|democrat|democrats|gop|maga|lawmaker|politician|"
    r"voter fraud|election fraud|stolen election|rigged election|"
    r"deep state|dnc|rnc|usaid|amway|"
    r"campaign donations?|@dnc|@rnc)\b",
    re.IGNORECASE,
)
BRAND_USERNAMES = {
    "lifelock", "aura", "norton", "nortononline", "mcafee",
    "cnn", "foxnews", "msnbc", "nytimes", "wsj", "reuters",
    "ap", "bbcworld", "abc", "abcnews", "nbcnews", "cbsnews",
}
EMOTIONAL_ONLY = re.compile(
    r"\b(catfish|catfished|catfishing|emotional scam|"
    r"scammed my heart|broke my heart)\b",
    re.IGNORECASE,
)

URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")
def clean_for_length(text):
    if not isinstance(text, str): return ""
    t = URL_RE.sub("", text)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+", " ", t).strip()

us_keywords = ["united states", "usa", "u.s.", " us ", " us,", "america",
    "new york", "california", "texas", "florida", "chicago",
    "los angeles", "boston", "seattle", "miami", "atlanta",
    "dallas", "houston", "philadelphia", "phoenix", "denver",
    "washington", "san francisco", "san diego",
    "ohio", "michigan", "georgia", "north carolina", "pennsylvania",
    "virginia", "arizona", "nevada", "oregon", "minnesota",
    "wisconsin", "tennessee", "indiana", "maryland", "massachusetts",
    "new jersey", "colorado", "kentucky", "alabama", "louisiana",
    "missouri", "iowa", "kansas", "arkansas", "oklahoma",
    " ny", " nj", " ca", " tx", " fl", " il", " pa", " oh", " va", " az",
]
def looks_us(row):
    if row.get("place_country_code") == "US": return True
    loc = " " + str(row.get("user_location") or "").lower() + " "
    return any(k in loc for k in us_keywords)


In [9]:
# Aplicar filtros sobre el corpus consolidado
df = df_dedup.copy()
df["clean_text"]       = df["text"].apply(clean_for_length)
df["clean_len"]        = df["clean_text"].str.len()
df["n_words"]          = df["clean_text"].str.split().str.len().fillna(0).astype(int)
df["hashtag_ratio"]    = (df["n_hashtags"] / df["n_words"].replace(0, 1)).fillna(0)
df["mention_ratio"]    = (df["n_mentions"] / df["n_words"].replace(0, 1)).fillna(0)
df["is_metaphor"]      = df["text"].fillna("").apply(lambda t: bool(METAPHOR_PATTERNS.search(t)))
df["is_soft_politics"] = df["text"].fillna("").apply(lambda t: bool(SOFT_POLITICS_PATTERNS.search(t)))
df["is_emotional"]     = df["text"].fillna("").apply(lambda t: bool(EMOTIONAL_ONLY.search(t)))
df["is_brand_account"] = df["username"].fillna("").str.lower().isin(BRAND_USERNAMES)
df["likely_us"]        = df.apply(looks_us, axis=1)
df["has_strong_fraud"] = df["text"].fillna("").apply(lambda t: bool(STRONG_FRAUD_TERMS.search(t)))
df["has_money"]        = df["text"].fillna("").apply(lambda t: bool(MONEY_TERMS.search(t)))
df["is_recovery_scammer"] = df["text"].fillna("").apply(is_recovery_scammer)
df["is_news_or_promo"]    = df.apply(lambda r: looks_like_news_or_promo(r["text"], r.get("n_urls", 0)), axis=1)
df["is_institutional_metaphor"] = df["text"].fillna("").apply(is_institutional_metaphor)
df["semantically_relevant"] = df["has_strong_fraud"] | df["has_money"]

mask = (
    (df["clean_len"] >= 40) &
    (df["semantically_relevant"]) &
    (df["likely_us"]) &
    (df["hashtag_ratio"] < 0.3) &
    (df["mention_ratio"] < 0.4) &
    (~df["is_metaphor"]) &
    (~df["is_soft_politics"]) &
    (~df["is_emotional"]) &
    (~df["is_brand_account"]) &
    (~df["is_recovery_scammer"]) &
    (~df["is_news_or_promo"]) &
    (~df["is_institutional_metaphor"])
)
df_clean = df[mask].reset_index(drop=True)

print("=" * 60)
print("   RESUMEN FINAL CONSOLIDADO")
print("=" * 60)
print(f"  Brutos unidos (run1+run2):  {len(df_all)}")
print(f"  Únicos:                     {len(df_dedup)}")
print(f"  Limpios estructurales:      {len(df_clean)}")
print(f"  Retención sobre únicos:     {len(df_clean)/max(len(df_dedup),1)*100:.1f}%")
print("=" * 60)
print("\nPor categoría temática:")
for label in QUERIES.keys():
    n = df_clean["query_labels"].fillna("").str.contains(label).sum()
    print(f"  {label:<20} {n:>6}")
print("\nDescartes:")
print(f"  Metáfora institucional:     {df['is_institutional_metaphor'].sum()}")
print(f"  Recovery scammers:          {df['is_recovery_scammer'].sum()}")
print(f"  Noticias/blog promos:       {df['is_news_or_promo'].sum()}")
print(f"  Política suave:             {df['is_soft_politics'].sum()}")
print(f"  Sin relevancia semántica:   {(~df['semantically_relevant']).sum()}")
print(f"  No-EEUU:                    {(~df['likely_us']).sum()}")


   RESUMEN FINAL CONSOLIDADO
  Brutos unidos (run1+run2):  12746
  Únicos:                     12726
  Limpios estructurales:      2324
  Retención sobre únicos:     18.3%

Por categoría temática:
  phishing                697
  payment_apps            189
  crypto                  778
  romance_monetary        167
  impersonation           500

Descartes:
  Metáfora institucional:     338
  Recovery scammers:          740
  Noticias/blog promos:       265
  Política suave:             140
  Sin relevancia semántica:   1317
  No-EEUU:                    9330


## Guardado final

In [10]:
df_clean.to_csv("../data/raw/scam_us_CONSOLIDATED_clean.csv", index=False, encoding="utf-8")
print(f"✓ Guardado: ../data/raw/scam_us_CONSOLIDATED_clean.csv ({len(df_clean)} filas)")
print()
print("📦 Archivos finales:")
print(f"   scam_us_run2_raw.csv             — bruto solo de esta tirada")
print(f"   scam_us_CONSOLIDATED_dedup.csv   — run1 + run2 deduplicado")
print(f"   scam_us_CONSOLIDATED_clean.csv   — filtrado final ({len(df_clean)} tweets)")
print()
print("🚨 HAZ COPIA DE SEGURIDAD del scam_us_CONSOLIDATED_clean.csv en Drive/Dropbox.")


✓ Guardado: ../data/raw/scam_us_CONSOLIDATED_clean.csv (2324 filas)

📦 Archivos finales:
   scam_us_run2_raw.csv             — bruto solo de esta tirada
   scam_us_CONSOLIDATED_dedup.csv   — run1 + run2 deduplicado
   scam_us_CONSOLIDATED_clean.csv   — filtrado final (2324 tweets)

🚨 HAZ COPIA DE SEGURIDAD del scam_us_CONSOLIDATED_clean.csv en Drive/Dropbox.


## Inspección visual

In [11]:
print("=== MUESTRA ALEATORIA DE 20 TWEETS LIMPIOS (CORPUS CONSOLIDADO) ===\n")
for _, row in df_clean.sample(min(20, len(df_clean)), random_state=42).iterrows():
    print(f"[{row['query_labels']}] @{row['username']} — {row['user_location']}")
    print(f"  {row['text'][:280]}")
    print()


=== MUESTRA ALEATORIA DE 20 TWEETS LIMPIOS (CORPUS CONSOLIDADO) ===

[crypto] @chriswithans — California, USA
  Bernie Madoff says there’s no evidence of a ponzi scheme at Bernard L. Madoff Investment Securities LLC https://t.co/mytXhaRK6j

[phishing] @ElijahWoodward9 — Arizona
  The Smishing Deluge: China-Based Campaign Flooding Global Text Messages
https://t.co/310HBvodxA

[crypto] @BellizaHanhzea — United States 
  Investment Scam Warning: 

#ApexFinancialInstitute is suspected of fraud and freezing user funds. Victims should contact trusted #CryptoRecovery experts for safe, reliable assistance. #CryptoScam #FundsRecovery #ScamAlert #InvestorAlert #Crypto https://t.co/vaDFC5vZag

[romance_monetary] @DatingDetectiv1 — New Jersey
  The DOJ seized $868K in crypto from a romance scam that lured victims into fake platforms through emotional manipulation—showing how love and money make a dangerous mix for cybercriminals. Learn more here: https://t.co/IDn2JLdzAH

[romance_monetary] @SuttiP

In [12]:
# 3 ejemplos por categoría para validar representatividad
print("=== 3 EJEMPLOS POR CATEGORÍA ===\n")
for label in QUERIES.keys():
    subset = df_clean[df_clean["query_labels"].fillna("").str.contains(label)]
    print(f"--- {label.upper()} ({len(subset)} tweets) ---")
    for _, row in subset.sample(min(3, len(subset)), random_state=42).iterrows():
        print(f"  @{row['username']} [{row.get('user_location', '')}]")
        print(f"    {row['text'][:240]}")
    print()


=== 3 EJEMPLOS POR CATEGORÍA ===

--- PHISHING (697 tweets) ---
  @cryptorecoveryh [United States]
    Scammers Target and Exploit Owners of Cryptocurrencies in Liquidity Mining Scam

Email: Cryptorecoveryhelpdesk@gmail.com if you’re a victim of crypto fraud 

#CryptoScam #CryptoRecovery #crypto #BTC
  @TeamLogicITOKC [Oklahoma City, OK]
    Have you fallen victim to a phishing email? Here's what to do if you respond to one. https://t.co/cri9KASfFn
  @MyTransitBoston [Boston, MA]
    Did you get the ‘EZDriveMA’ text? MassDOT says don’t fall for this smishing scam https://t.co/LYzc8t7W6E #Boston #Transit #MBTA #MassDOT #BostonNews

--- PAYMENT_APPS (189 tweets) ---
  @FoulMouthLeftie [United States]
    🚨Scam profile alert🚨Paula Wilson (paulawilson00). Her profile is attempting to target people when they engage with her with. Cash app scam. I’ve reported her. Block her!
  @van_taylan [USA]
    Attorney General Sunday Warns Consumers to be Aware of Trending PayPal Scam

https://t.co/jxpx